# Search lookup magics

Use **`%%cribl_save_search_lookup`** / **`%%cribl_load_search_lookup`** / **`%%cribl_delete_search_lookup`** for Search lookup CSV files under the **`default_search`** API group (same as `%%cribl_search`).

**Limits:** lookups are capped at **10,000 rows**. Use **`replace=true`** on save to overwrite. Pick **distinct filenames** (for example `notebook_app_lookup_demo.csv`) so you do not clash with shared lookups.

This notebook also shows **querying lookup contents** with the `$vt_lookups` virtual table (`lookupFile` is the **stem** without `.csv`; see Cribl `$vt_lookups` docs), and **creating then deleting** a separate lookup using **`pyodide.http.pyfetch`** (iframe fetch bridge / auth) plus an optional **`cribl-control-plane`** health check.

## Setup

Run **once** before other code cells. Matches `Cribl_Python_SDK.ipynb` pins for Pyodide WASM wheels.


In [ ]:
import micropip

# Match Pyodide 0.29.x built-in stack (WASM). Update when bumping PYODIDE_RELEASE.
# `httpx` is required by `cribl-control-plane`; CRIBL REST in this notebook uses `pyodide.http.pyfetch`.
await micropip.install([
    'httpcore==1.0.7',
    'httpx==0.28.1',
    'pydantic-core==2.41.5',
    'pydantic==2.12.5',
    'cribl-control-plane',
])


## A) Cell magics: Search → save → search via `$vt_lookups` → load → verify


In [ ]:
%%cribl_search var=sample_df preview=false
dataset=cribl_search_sample | sort by _time desc | limit 50


In [ ]:
%%cribl_save_search_lookup notebook_app_lookup_demo.csv var=sample_df replace=true


In [ ]:
%%cribl_search var=vt_rows preview=true
dataset="$vt_lookups" lookupFile="notebook_app_lookup_demo"
| limit 25


In [ ]:
%%cribl_load_search_lookup notebook_app_lookup_demo.csv var=from_lookup


In [ ]:
print('search rows:', len(sample_df))
print('vt_lookups rows:', len(vt_rows))
print('loaded rows:', len(from_lookup))
from_lookup.head(3)


## B) Create & delete a lookup with Python (`pyfetch` + `cribl-control-plane`)

In the Cribl Apps iframe, **Pyodide routes `fetch` through the parent** so CRIBL API calls get auth and proxy rules. Third-party **`httpx`** calls from Python often **bypass** that bridge and fail (401 / network). **`pyodide.http.pyfetch`** uses the same patched `fetch`, so it matches the behavior of `%%cribl_api` / lookup magics.

This cell still calls **`CriblControlPlane.health`** (uses `httpx` inside the SDK) for a quick smoke check; if that part errors in your tenant, rely on the `pyfetch` block for lookup REST.


In [ ]:
import json
import os
from urllib.parse import quote

from cribl_control_plane import CriblControlPlane
from pyodide.http import pyfetch

base = os.environ['CRIBL_API_URL'].rstrip('/')
group = 'default_search'
lookup_name = 'notebook_app_sdk_lookup_demo.csv'
csv_body = 'join_key,label\nalpha,from_pyfetch\nbeta,second_row\n'

print('SDK health ping:', end=' ')
try:
    with CriblControlPlane(server_url=base) as ccp:
        print(ccp.health.get())
except Exception as exc:
    print('(skipped — SDK uses httpx which may not use the iframe fetch bridge)', repr(exc))

put_url = f'{base}/m/{group}/system/lookups?filename=__nb_sdk_tmp.csv'
r_put = await pyfetch(
    put_url,
    method='PUT',
    body=csv_body.encode('utf-8'),
    headers={'Content-Type': 'text/csv'},
)
if r_put.status >= 400:
    raise RuntimeError(f'PUT upload failed {r_put.status}: {await r_put.text()}')
tmp = await r_put.json()
tmp_name = tmp['filename']

r_post = await pyfetch(
    f'{base}/m/{group}/system/lookups',
    method='POST',
    body=json.dumps({'id': lookup_name, 'fileInfo': {'filename': tmp_name}}),
    headers={'Content-Type': 'application/json'},
)
if r_post.status >= 400:
    raise RuntimeError(f'POST register failed {r_post.status}: {await r_post.text()}')

qn = quote(lookup_name, safe='')
r_del = await pyfetch(f'{base}/m/{group}/system/lookups/{qn}', method='DELETE')
if r_del.status not in (200, 204, 404):
    raise RuntimeError(f'DELETE failed {r_del.status}: {await r_del.text()}')
print('pyfetch create/delete finished for', lookup_name)


## Cleanup (cell magics)

Deletes the lookup created in section **A** (`notebook_app_lookup_demo.csv`). Section **B** already removed its own file.


In [ ]:
%%cribl_delete_search_lookup notebook_app_lookup_demo.csv
